# Clase 2 — Tu primer análisis de datos
### Maestría en Fintech · ITBA · 2026

---

**Pregunta que vamos a responder hoy:**
> *¿Qué tipo de cliente de una fintech tiene más probabilidad de irse?*

**Dataset:** 1.000 clientes de una fintech argentina ficticia

| Columna | Descripción |
|---|---|
| `cliente_id` | Identificador único |
| `nombre` | Nombre del cliente |
| `edad` | Edad en años |
| `provincia` | Provincia de residencia |
| `segmento` | Premium / Retail / PyME / Joven |
| `antiguedad_años` | Años como cliente |
| `balance_ars` | Saldo en cuenta (ARS) |
| `cant_productos` | Cantidad de productos contratados |
| `cliente_activo` | Si opera actualmente |
| `churn` | **1 = se fue · 0 = se quedó** |

---
## Setup

No hace falta subir ningún archivo. El notebook descarga el dataset directamente desde el repositorio del curso: se ejecuta la celda y listo.

Si en algún momento querés trabajar con tu propio CSV, subilo desde el panel izquierdo de Colab (ícono de carpeta → subir archivo) y reemplazá la URL por el nombre del archivo.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Librerías cargadas correctamente')

---
## 1. Cargar el dataset

La semana pasada vimos que una **librería** es un conjunto de herramientas que alguien ya escribió para nosotros, y que `import pandas as pd` carga esa librería con el apodo `pd`.

Ahora la usamos por primera vez.

In [ ]:
# Los datos se descargan solos desde el repo del curso.
# No hace falta subir ningún archivo a Colab.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

df = pd.read_csv(DATOS + 'clientes.csv')

# Primeras 5 filas
df.head()


`pd.read_csv()` lee el archivo y lo convierte en un **DataFrame**: la tabla de Python.  
`df` es el nombre de la variable — convención, no obligación. Podrían llamarla `clientes`, `tabla`, lo que quieran.

In [ ]:
# Últimas 5 filas — para verificar que el CSV cargó completo
df.tail()

In [ ]:
# Dimensiones: (filas, columnas)
df.shape

---
## 2. Exploración inicial

Antes de hacer cualquier análisis, necesitamos entender qué tenemos: qué tipo de datos, si hay celdas vacías, qué rango tienen los números.

In [ ]:
# Tipos de datos y valores nulos por columna
df.info()

**Lo que nos dice `.info()`:**
- El tipo de cada columna: `int64` (entero), `float64` (decimal), `object` (texto), `bool` (verdadero/falso)
- Si hay celdas vacías: si el conteo `non-null` es igual al total de filas, no hay nulos

En la **Clase 5** vamos a trabajar con un dataset con nulos, duplicados y outliers deliberados. Por ahora, este dataset está limpio.

In [ ]:
# Estadísticas descriptivas de todas las columnas numéricas
df.describe().round(2)

**Lo que nos dice `.describe()`:**
- `mean` → promedio
- `50%` → mediana (el valor que divide la distribución a la mitad)
- `std` → desvío estándar (qué tan dispersos están los valores)
- `min` / `max` → extremos

> **Dato directo:** la columna `churn` tiene valores 0 y 1, entonces su `mean` es exactamente la tasa de churn de la cartera.

---
## 3. Selección de columnas

In [ ]:
# Una columna → devuelve una Serie
df['segmento']

In [ ]:
# Varias columnas → devuelve un DataFrame
df[['nombre', 'segmento', 'balance_ars']].head()

In [ ]:
# Valores únicos de una columna categórica
df['segmento'].unique()

In [ ]:
# Cuántos clientes hay por segmento — ordenados de mayor a menor
df['segmento'].value_counts()

---
## 4. Filtros: hacerle preguntas a los datos

En Excel filtraban con el botón de filtro y hacían clic en los valores que querían.  
En Python, el filtro es una **condición escrita** — reproducible, automática, sin clics.

La lógica: la condición devuelve `True` o `False` para cada fila. Python se queda solo con las filas donde es `True`.

In [ ]:
# Clientes que se fueron (churn = 1)
df[df['churn'] == 1]

In [ ]:
# ¿Cuántos se fueron?
df[df['churn'] == 1].shape[0]

In [ ]:
# Guardar el filtro en una variable para reutilizarlo
churn = df[df['churn'] == 1]
activos = df[df['churn'] == 0]

print(f'Clientes que se fueron:    {len(churn)}')
print(f'Clientes que se quedaron:  {len(activos)}')

### Filtros compuestos

- `&` → Y (ambas condiciones deben ser verdaderas)
- `|` → O (al menos una debe ser verdadera)
- Los paréntesis alrededor de cada condición son **obligatorios**

In [ ]:
# Clientes Premium activos
df[(df['segmento'] == 'Premium') & (df['cliente_activo'] == True)]

In [ ]:
# Clientes con balance alto O mucha antigüedad
df[(df['balance_ars'] > 500000) | (df['antiguedad_años'] > 5)]

In [ ]:
# Clientes Jóvenes que se fueron
df[(df['segmento'] == 'Joven') & (df['churn'] == 1)]

---
## 5. Estadísticas descriptivas en contexto de negocio

Los números solos no dicen nada. La pregunta siempre es: *¿qué significa este número para el negocio?*

In [ ]:
# Estadísticas básicas de balance
print(f"Promedio:  ${df['balance_ars'].mean():>12,.0f}")
print(f"Mediana:   ${df['balance_ars'].median():>12,.0f}")
print(f"Desvío:    ${df['balance_ars'].std():>12,.0f}")
print(f"Mínimo:    ${df['balance_ars'].min():>12,.0f}")
print(f"Máximo:    ${df['balance_ars'].max():>12,.0f}")

> **Promedio vs mediana:** si el promedio es mucho mayor que la mediana, hay clientes con balances muy altos que distorsionan el promedio. En carteras de clientes, siempre mirar los dos.

In [ ]:
# Balance promedio: ¿quiénes se van vs quiénes se quedan?
df.groupby('churn')['balance_ars'].mean().apply(lambda x: f'${x:,.0f}')

In [ ]:
# Tasa de churn por segmento — ordenada de mayor a menor
df.groupby('segmento')['churn'].mean().sort_values(ascending=False).apply(lambda x: f'{x:.1%}')

> `groupby` agrupa el DataFrame por los valores de una columna y aplica una función a otra.  
> Lo vamos a ver en profundidad en la **Clase 3**.

---
## 6. Primeras visualizaciones

El objetivo no es hacer gráficos bonitos — es que el gráfico nos diga algo que los números solos no comunican de forma inmediata.

In [ ]:
# Distribución de edades de la cartera
df['edad'].hist(bins=20, color='steelblue', edgecolor='white')
plt.title('Distribución de edades')
plt.xlabel('Edad')
plt.ylabel('Cantidad de clientes')
plt.show()

**¿Qué nos dice este gráfico?** ¿La cartera es joven o madura? ¿Hay un rango de edad dominante? ¿Eso es consistente con el producto que ofrece la fintech?

In [ ]:
# Clientes por segmento
df['segmento'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Clientes por segmento')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.show()

---
## 7. Caso práctico: ¿qué le decimos al gerente de retención?

Tres preguntas concretas. Los números que siguen son el insumo para una recomendación de negocio.

In [ ]:
# Pregunta 1: ¿Cuál es la tasa de churn global?
tasa_churn = df['churn'].mean()
print(f'Tasa de churn global: {tasa_churn:.1%}')

In [ ]:
# Pregunta 2: ¿Qué segmento tiene más churn?
churn_por_segmento = df.groupby('segmento')['churn'].mean().sort_values(ascending=False)

churn_por_segmento.sort_values().plot(kind='barh', color='salmon', edgecolor='white')
plt.title('Tasa de churn por segmento')
plt.xlabel('Tasa de churn')
plt.show()

print(churn_por_segmento.apply(lambda x: f'{x:.1%}').to_string())

In [ ]:
# Pregunta 3: ¿El balance de quienes se van es distinto al de quienes se quedan?
balance_por_churn = df.groupby('churn')['balance_ars'].mean()

print('Balance promedio:')
print(f"  Clientes que se quedaron: ${balance_por_churn[0]:>12,.0f}")
print(f"  Clientes que se fueron:   ${balance_por_churn[1]:>12,.0f}")

### Del análisis a la recomendación

Con estos tres números podemos armar el mensaje para el gerente de retención:

1. **El problema:** X% de los clientes se fue en el período analizado
2. **El segmento crítico:** El segmento [X] tiene una tasa de churn de Y% — [N]x mayor que el promedio
3. **El perfil de riesgo:** Los clientes que se van tienen un balance promedio de $Z — [mayor/menor] que quienes se quedan

> Un análisis = datos + interpretación + recomendación

---
## Ejercicios

Usá el mismo dataset `df` para responder las siguientes preguntas.  
Podés usar Gemini si necesitás ayuda con la sintaxis — el objetivo es responder la pregunta de negocio, no memorizar el código.

### Ejercicio 1
¿Cuántos clientes hay por provincia? Mostralo ordenado de mayor a menor.

In [ ]:
# Tu código acá


### Ejercicio 2
¿Cuál es la edad promedio de los clientes por segmento?

In [ ]:
# Tu código acá


### Ejercicio 3
Filtrá los clientes de Buenos Aires con más de 3 productos contratados. ¿Cuántos son?

In [ ]:
# Tu código acá


### Ejercicio 4
¿Cuál es el balance máximo, mínimo y mediano de toda la cartera?

In [ ]:
# Tu código acá


### Ejercicio 5
¿Qué porcentaje de clientes está activo (`cliente_activo == True`)?

In [ ]:
# Tu código acá


### Ejercicio 6
Filtrá los clientes del segmento Joven con churn igual a 1. ¿Cuál es su antigüedad promedio comparada con los Jóvenes que se quedaron?

In [ ]:
# Tu código acá


### Ejercicio 7
¿Cuántos clientes tienen más de 5 años de antigüedad y no tienen churn?

In [ ]:
# Tu código acá


### Ejercicio 8
Hacé un histograma de la distribución de `balance_ars`. ¿Qué forma tiene? ¿Qué te dice eso sobre la cartera?

In [ ]:
# Tu código acá


### Ejercicio 9
¿En qué provincias hay mayor tasa de churn? Mostrá las 5 provincias con más churn.

In [ ]:
# Tu código acá


### Ejercicio 10 — Desafío
Construí un gráfico de barras que muestre el **balance promedio por segmento**, ordenado de mayor a menor.  
Abajo del gráfico, escribí en un comentario qué conclusión de negocio sacás.

In [ ]:
# Tu código acá


# Tu conclusión:
# 